In [ ]:
# Contributors: Antonio Meneses, Julián Consuegra, Juan Pablo Miró-Quesada, Tomás Roschge
# Course: Recommendation Engines · IE University · 2025-26
# Notebook 02: Collaborative Filtering (SVD)

In [ ]:
import sys, os

_nb_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.path.join(_nb_dir, '..'),
    os.path.join(os.getcwd(), '..'),
    '/content/group5-rec-engines',
    os.getcwd(),
]
for _root in _candidates:
    if os.path.isfile(os.path.join(_root, 'src', 'utils.py')):
        sys.path.insert(0, os.path.join(_root, 'src'))
        os.chdir(os.path.join(_root, 'notebooks'))
        break
else:
    raise FileNotFoundError("Cannot find src/utils.py. Run from the repo root or notebooks/ directory.")

from utils import *

In [ ]:
import gc
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from surprise import Dataset, Reader, SVD, accuracy

np.random.seed(42)

## 1. Data Loading & Train/Test Split

In [ ]:
df, df_meta = load_data()
print(f'Reviews loaded: {len(df):,}')

In [ ]:
train_df, test_df, split_info = temporal_train_test_split(df)
train_users = split_info['train_users']
train_items = split_info['train_items']
cold_users  = split_info['cold_users']
cold_items  = split_info['cold_items']

In [ ]:
lookups = build_lookup_structures(train_df, test_df)
GLOBAL_MEAN      = lookups['global_mean']
train_user_items = lookups['train_user_items']

sample_users = sample_eval_users(lookups, n=1000)

## 2. Collaborative Filtering — SVD

We use SVD matrix factorization via **scikit-surprise**:
- 30 latent factors, 20 training epochs, learning rate 0.005, regularisation 0.02
- Item-based KNNBaseline is skipped: with 241K users the similarity matrix exceeds available RAM.

SVD decomposes the user-item rating matrix into latent factor matrices that capture complex
user-item relationships from the collaborative signal alone (no metadata required).

In [ ]:
reader = Reader(rating_scale=(1.0, 5.0))
train_surprise = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
).build_full_trainset()

test_surprise = [
    (row.user_id, row.item_id, row.rating)
    for row in test_df.itertuples()
]

print(f'Trainset: {train_surprise.n_users:,} users, {train_surprise.n_items:,} items')

In [ ]:
gc.collect()
print('Training SVD (n_factors=30, n_epochs=20)...')
t0 = time.time()
algo_svd = SVD(n_factors=30, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_svd.fit(train_surprise)
print(f'  Done in {time.time()-t0:.1f}s')
gc.collect()

## 3. Rating Prediction

In [ ]:
print('Evaluating SVD on test set...')
preds_svd = algo_svd.test(test_surprise)
rmse_svd  = accuracy.rmse(preds_svd, verbose=False)
mae_svd   = accuracy.mae(preds_svd, verbose=False)
print(f'  SVD — RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f}')

## 4. Ranking Metrics

In [ ]:
def get_cf_top_k(algo, user_id, k=K):
    """Top-K recommendations via SVD. Samples 2000 candidates for efficiency
    (scoring all 134K train items would take ~2 min per user)."""
    if algo is None:
        return []
    seen       = train_user_items.get(user_id, set())
    candidates = [it for it in train_items if it not in seen]
    if len(candidates) > 2000:
        candidates = list(np.random.choice(candidates, 2000, replace=False))
    scores = [(it, algo.predict(user_id, it).est) for it in candidates]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [it for it, _ in scores[:k]]

In [ ]:
print('Computing ranking metrics for SVD (1000 users)...')
fn_svd = lambda uid: get_cf_top_k(algo_svd, uid)
all_recs_svd, ranking_svd = evaluate_ranking(fn_svd, sample_users, lookups)

recs_svd_dict = {u: r for u, r in zip(sample_users, all_recs_svd)}

results_cf = {
    'SVD': {'RMSE': rmse_svd, 'MAE': mae_svd, 'Diversity': None, **ranking_svd}
}
print_metrics('SVD (CF)', rmse_svd, mae_svd, ranking_svd)

## 5. Cold-Start Analysis

In [ ]:
warm_preds      = [p for p in preds_svd if p.uid in train_users and p.iid in train_items]
cold_user_preds = [p for p in preds_svd if p.uid not in train_users]
cold_item_preds = [p for p in preds_svd if p.iid not in train_items]

cold_data = {
    'Warm':      rmse([p.r_ui for p in warm_preds],      [p.est for p in warm_preds])      if warm_preds else 0,
    'Cold User': rmse([p.r_ui for p in cold_user_preds], [p.est for p in cold_user_preds]) if cold_user_preds else 0,
    'Cold Item': rmse([p.r_ui for p in cold_item_preds], [p.est for p in cold_item_preds]) if cold_item_preds else 0,
}
for cat, val in cold_data.items():
    count = len(warm_preds if cat == 'Warm' else (cold_user_preds if cat == 'Cold User' else cold_item_preds))
    print(f'  {cat:12s}: RMSE={val:.4f}  ({count:,} predictions)')

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs actual rating distributions
svd_actual = [p.r_ui for p in preds_svd[:50000]]
svd_pred   = [p.est  for p in preds_svd[:50000]]
axes[0].hist(svd_actual, bins=20, alpha=0.6, label='Actual',         color='steelblue', edgecolor='white')
axes[0].hist(svd_pred,   bins=20, alpha=0.6, label='Predicted (SVD)', color='coral',    edgecolor='white')
axes[0].set_title('SVD: Actual vs Predicted Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].legend()

# Cold-start RMSE
axes[1].bar(cold_data.keys(), cold_data.values(),
            color=['seagreen', 'coral', 'steelblue'], edgecolor='white')
axes[1].set_title('SVD: RMSE by Cold-Start Category')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
import os, pickle

output = {
    'results_cf':   results_cf,
    'cold_data':    cold_data,
    'recs_cf':      {'SVD': recs_svd_dict},   # {user_id: [item_ids]}
    'sample_users': sample_users,
}

os.makedirs('../results', exist_ok=True)
with open('../results/02_collaborative_filtering.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Saved: ../results/02_collaborative_filtering.pkl')